# Hai việc: ① tinh chỉnh trên ô cắt ortho  ② `yolov8s-obb` vòng 1–4

**① Tinh chỉnh ortho — việc chính.** Bài báo giữ cố định model/trọng số/ngưỡng rồi so hai hướng xử lý.
Phản biện sẽ hỏi: *model train trên khung, đương nhiên kém trên ortho — train lại trên ortho thì sao?*
Ô này khởi từ `best.pt` của từng vòng B3 rồi tinh chỉnh trên **ô cắt ortho có nhãn**, dựng bằng
`ma/cat_o_ortho_train.py`: nhãn tay trên 497 khung chiếu xuống mặt đất, gộp trùng 0,60 m còn **727 hộp**,
cắt cửa sổ 640×512 chồng 50 %, xám → INFERNO, xoay 270° — **đúng thứ model sẽ thấy lúc chấm**.

Làm theo kiểu **tinh chỉnh chứ không train từ đầu**: ortho chỉ có một tấm, 727 hộp, so với 12.390 hộp của
nhánh khung. Train mới rồi thua thì phản biện quy cho thiếu mẫu, phép so hỏng. Tinh chỉnh thì luận điểm
thành *đã cho nhánh ghép ưu thế sân nhà mà vẫn thua*.

**Vai của ô theo vòng** lấy từ `chia_o_ortho.csv`: ô tra khung TN19 gần nhất, **tâm ô và 4 góc ô phải cùng
một vai**, lệch nhau thì ô thành `dem`. Không có thế thì ô vắt qua ranh giới khối sẽ kéo vùng test vào train.
`val` dùng ô `dem` — không nằm trong test nên không rò rỉ vào con số của bài.

| vòng | ô train | ô val (`dem`) | ô test |
|---|---|---|---|
| 0 | 139 | 64 | 44 |
| 1 | 133 | 114 | **0** |
| 2 | 115 | 108 | 24 |
| 3 | 116 | 116 | 15 |
| 4 | 128 | 67 | 52 |

Vòng 1 không có ô test vì vùng bay của ortho không giao với khối test vòng 1 — đúng như lượt quét
out-of-fold hiện có, vòng đó không được dùng trên ortho. Ô 4 tự bỏ qua.

**② `yolov8s-obb` vòng 1–4** — bộ dò thứ hai chạy đủ out-of-fold, để trả lời "ΔR là của ảnh hay của riêng
`yolo26s-obb`". Đọc thẳng `khung_nhan.zip` + `chia_khoi.csv` đã có sẵn ở `MyDrive/colab_B`, **không phải
tải lại 634 MB**. Kết quả ghi về `MyDrive/colab_B/ket_qua` để `ma/gom_bangA_B3.py` gom được như cũ.

**Trước khi chạy.** `MyDrive/colab_C/` có `o_ortho.zip`, `goc_pt.zip`, `chia_o_ortho.csv`, notebook này;
`MyDrive/colab_B/` giữ nguyên như đợt trước. Runtime → GPU.

**Ước lượng T4:** ô 4 khoảng 10–15 phút/vòng × 5 ≈ **1 giờ**; ô 5 khoảng 50–70 phút/lượt × 4 ≈ **4 giờ**.
Lượt nào xong là ghi JSON ngay, đứt phiên chạy lại ô 1→3 rồi ô 4/5, lượt nào có JSON thì bỏ qua.


In [1]:
# 1. Drive + giải nén + cài
from google.colab import drive; drive.mount('/content/drive')
import os, zipfile, json, csv, time, glob, shutil
C = '/content/drive/MyDrive/colab_C'
B = '/content/drive/MyDrive/colab_B'
assert os.path.isdir(C), C
assert os.path.isdir(B), B + '  (ô 5 cần khung_nhan.zip + chia_khoi.csv của đợt trước)'
KQ  = f'{C}/ket_qua';  os.makedirs(KQ, exist_ok=True)     # kết quả ô cắt ortho
KQB = f'{B}/ket_qua';  os.makedirs(KQB, exist_ok=True)    # kết quả khung, gom chung với đợt trước

if not os.path.isdir('/content/o_ortho/images'):
    t = time.time(); zipfile.ZipFile(f'{C}/o_ortho.zip').extractall('/content/o_ortho')
    print('giải nén ô cắt %.0f s' % (time.time() - t))
if not os.path.isdir('/content/goc'):
    zipfile.ZipFile(f'{C}/goc_pt.zip').extractall('/content/goc')
O_ROWS = list(csv.DictReader(open(f'{C}/chia_o_ortho.csv', encoding='utf8')))
print(len(O_ROWS), 'ô cắt ortho |', len(os.listdir('/content/o_ortho/images')), 'ảnh |',
      len(glob.glob('/content/goc/*.pt')), 'trọng số gốc')

!pip -q install ultralytics
import ultralytics; print('ultralytics', ultralytics.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


Mounted at /content/drive
giải nén ô cắt 4 s
247 ô cắt ortho | 247 ảnh | 5 trọng số gốc
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
ultralytics 8.4.160
Tesla T4, 15360 MiB


In [2]:
# 2. Dựng thư mục ô cắt ortho theo vòng. train = vai train, val = ô `dem`, test = vai test.
LOP = ['SingleHotSpot', 'MultiHotSpot', 'Bypass', 'ModuleOpenCircuit']
K = 5

def dung_ortho(vong):
    goc = f'/content/ds_ortho_v{vong}'
    if os.path.isdir(goc): return goc
    for vai in ('train', 'val', 'test'):
        os.makedirs(f'{goc}/{vai}/images'); os.makedirs(f'{goc}/{vai}/labels')
    for r in O_ROWS:
        v = r[f'vai_v{vong}']
        vai = {'train': 'train', 'dem': 'val', 'test': 'test'}.get(v)
        if vai is None: continue
        ten = r['file'][:-4]
        os.symlink(f'/content/o_ortho/images/{r["file"]}', f'{goc}/{vai}/images/{r["file"]}')
        os.symlink(f'/content/o_ortho/labels/{ten}.txt',   f'{goc}/{vai}/labels/{ten}.txt')
    open(f'{goc}/data.yaml', 'w').write(
        f'path: {goc}\ntrain: train/images\nval: val/images\ntest: test/images\nnc: 4\nnames: {LOP}\n')
    return goc

for v in range(K):
    g = dung_ortho(v)
    n = {vai: len(os.listdir(f'{g}/{vai}/images')) for vai in ('train', 'val', 'test')}
    print(f'vòng {v}: {n}')


vòng 0: {'train': 139, 'val': 64, 'test': 44}
vòng 1: {'train': 133, 'val': 114, 'test': 0}
vòng 2: {'train': 115, 'val': 108, 'test': 24}
vòng 3: {'train': 116, 'val': 116, 'test': 15}
vòng 4: {'train': 128, 'val': 67, 'test': 52}


In [3]:
# 3. Hàm tinh chỉnh + chấm. Chấm CẢ model gốc lẫn model đã tinh chỉnh trên cùng ô test -> cặp trước/sau.
from ultralytics import YOLO
import numpy as np

# Giống hệt cấu hình B3, đổi đúng hai chỗ: epochs 150->60 và lr0 0,005->0,001 (tinh chỉnh, không train mới).
CAU_HINH_TC = dict(epochs=60, patience=20, batch=16, imgsz=640, lr0=0.001, degrees=15,
                   flipud=0.5, fliplr=0.5, mosaic=1.0, close_mosaic=15, optimizer='auto',
                   workers=2, plots=False, verbose=False)

def theo_lop(b):
    ra = [None] * 4
    for k, ci in enumerate(b.ap_class_index):
        ra[int(ci)] = round(float(b.ap50[k]), 4)
    return ra

def cham(m, dy, split='test'):
    r = m.val(data=dy, split=split, plots=False, verbose=False)
    b = r.box
    return dict(mAP50=round(float(b.map50), 4), mAP50_95=round(float(b.map), 4), AP50_theo_lop=theo_lop(b))

def mot_vong_ortho(vong, hat=0):
    tag = f'yolo26s-obb_v{vong}_s{hat}'
    ra = f'{KQ}/ortho_tc_{tag}.json'
    if os.path.exists(ra):
        print('  bỏ qua, đã có:', tag); return json.load(open(ra))
    goc = dung_ortho(vong); dy = f'{goc}/data.yaml'
    n_test = len(os.listdir(f'{goc}/test/images'))
    p_goc = f'/content/goc/{tag}.pt'
    run = f'{KQ}/runs_ortho/{tag}'
    t0 = time.time()

    truoc = cham(YOLO(p_goc), dy) if n_test else None   # model khung, chưa tinh chỉnh, trên ô test ortho

    last = f'{run}/weights/last.pt'
    if os.path.exists(last):
        print('  nối tiếp từ last.pt:', tag); m = YOLO(last); m.train(resume=True)
    else:
        m = YOLO(p_goc)
        m.train(data=dy, seed=hat, project=f'{KQ}/runs_ortho', name=tag, exist_ok=True, **CAU_HINH_TC)

    sau = cham(YOLO(f'{run}/weights/best.pt'), dy) if n_test else None
    kq = dict(vong=vong, hat=hat, goc=os.path.basename(p_goc),
              o_train=len(os.listdir(f'{goc}/train/images')),
              o_val=len(os.listdir(f'{goc}/val/images')), o_test=n_test,
              phut=round((time.time() - t0) / 60, 1),
              epoch_chay=int(len(open(f'{run}/results.csv').read().splitlines()) - 1),
              truoc=truoc, sau=sau, cau_hinh=CAU_HINH_TC)
    json.dump(kq, open(ra, 'w'), ensure_ascii=False, indent=1)
    if n_test:
        print(f'  {tag}: mAP50 trên ô test ortho  trước {truoc["mAP50"]:.4f} -> sau {sau["mAP50"]:.4f}'
              f'   ({kq["phut"]} phút, {kq["epoch_chay"]} epoch)')
    else:
        print(f'  {tag}: không có ô test ở vòng này, chỉ lưu trọng số ({kq["phut"]} phút)')
    return kq


In [ ]:
# 4. ① Tinh chỉnh ortho, 5 vòng. Trọng số ra: {KQ}/runs_ortho/yolo26s-obb_v{k}_s0/weights/best.pt
for v in range(K):
    print(f'=== tinh chỉnh ortho  vòng {v} ===')
    mot_vong_ortho(v, 0)

print('\n--- bảng trước/sau trên ô test ortho ---')
for v in range(K):
    p = f'{KQ}/ortho_tc_yolo26s-obb_v{v}_s0.json'
    if not os.path.exists(p): continue
    d = json.load(open(p))
    if d['truoc'] is None:
        print(f'  vòng {v}: không có ô test'); continue
    print(f'  vòng {v}: {d["o_test"]:>3} ô test | mAP50 {d["truoc"]["mAP50"]:.4f} -> {d["sau"]["mAP50"]:.4f}'
          f'  ({d["sau"]["mAP50"] - d["truoc"]["mAP50"]:+.4f})')


=== tinh chỉnh ortho  vòng 0 ===
Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s-obb summary (fused): 129 layers, 9,752,715 parameters, 0 gradients, 21.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1639.8±1023.7 MB/s, size: 140.5 KB)
val: Scanning /content/ds_ortho_v0/test/labels... 44 images, 9 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 44/44 524.3it/s 0.1s
val: New cache created: /content/ds_ortho_v0/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.4it/s 1.3s
                   all         44        234      0.687      0.538      0.455      0.406
Speed: 1.0ms preprocess, 10.5ms inference, 0.0ms loss, 7.0ms postprocess per image
Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, 

In [4]:
# 5. ② yolov8s-obb vòng 1..4 trên KHUNG — đọc dữ liệu của colab_B, ghi kết quả về colab_B/ket_qua.
if not os.path.isdir('/content/khung/images'):
    t = time.time(); zipfile.ZipFile(f'{B}/khung_nhan.zip').extractall('/content/khung')
    print('giải nén khung %.0f s' % (time.time() - t))
K_ROWS = list(csv.DictReader(open(f'{B}/chia_khoi.csv', encoding='utf8')))

def dung_khung(vong):
    goc = f'/content/ds_obb_v{vong}'
    if os.path.isdir(goc): return goc
    for vai in ('train', 'val', 'test'):
        os.makedirs(f'{goc}/{vai}/images'); os.makedirs(f'{goc}/{vai}/labels')
    for r in K_ROWS:
        vai = r[f'vai_v{vong}']
        if vai == 'dem': continue
        ten = r['file'][:-4]
        os.symlink(f'/content/khung/images/{r["file"]}', f'{goc}/{vai}/images/{r["file"]}')
        os.symlink(f'/content/khung/labels/{ten}.txt',   f'{goc}/{vai}/labels/{ten}.txt')
    open(f'{goc}/data.yaml', 'w').write(
        f'path: {goc}\ntrain: train/images\nval: val/images\ntest: test/images\nnc: 4\nnames: {LOP}\n')
    return goc

# Cấu hình khoá y hệt đợt trước, chỉ đổi patience 20 -> 50 (đợt trước hai hạt bị cắt ở 56/43 epoch).
CAU_HINH_B = dict(epochs=150, patience=50, batch=16, imgsz=640, lr0=0.005, degrees=15,
                  flipud=0.5, fliplr=0.5, mosaic=1.0, close_mosaic=15, optimizer='auto',
                  workers=4, plots=False, verbose=False)

def theo_lop2(b, thuoc):
    v = getattr(b, thuoc); ra = [None] * 4
    for k, ci in enumerate(b.ap_class_index):
        ra[int(ci)] = round(float(v[k]), 4)
    return ra

def cham_conf(m, dy, conf):
    b = m.val(data=dy, split='test', conf=conf, iou=0.5, plots=False, verbose=False).box
    return dict(P_theo_lop=theo_lop2(b, 'p'), R_theo_lop=theo_lop2(b, 'r'), mAP50=round(float(b.map50), 4))

def mot_luot_khung(ten, vong, hat):
    tag = f'{ten}_v{vong}_s{hat}'
    ra = f'{KQB}/{tag}.json'
    if os.path.exists(ra):
        print('  bỏ qua, đã có:', tag); return json.load(open(ra))
    goc = dung_khung(vong); dy = f'{goc}/data.yaml'
    run = f'{KQB}/runs/{tag}'; last = f'{run}/weights/last.pt'
    t0 = time.time()
    if os.path.exists(last):
        print('  nối tiếp từ last.pt:', tag); m = YOLO(last); m.train(resume=True)
    else:
        m = YOLO(f'{ten}.yaml')
        m.train(data=dy, seed=hat, project=f'{KQB}/runs', name=tag, exist_ok=True, **CAU_HINH_B)
    m = YOLO(f'{run}/weights/best.pt')
    b = m.val(data=dy, split='test', plots=False, verbose=False).box
    kq = dict(kien_truc=ten, yaml=f'{ten}.yaml', kieu_hop='obb', vong=vong, hat=hat,
              tham_so_M=round(sum(p.numel() for p in m.model.parameters()) / 1e6, 2),
              epoch_chay=int(len(open(f'{run}/results.csv').read().splitlines()) - 1),
              phut=round((time.time() - t0) / 60, 1),
              test=dict(mAP50=round(float(b.map50), 4), mAP50_95=round(float(b.map), 4),
                        AP50_theo_lop=theo_lop2(b, 'ap50'), lop=LOP),
              conf_0_25=cham_conf(m, dy, 0.25), conf_0_5=cham_conf(m, dy, 0.5),
              cau_hinh=CAU_HINH_B)
    json.dump(kq, open(ra, 'w'), ensure_ascii=False, indent=1)
    print(f'  {tag}: mAP50 {kq["test"]["mAP50"]:.4f}  AP50 theo lớp {kq["test"]["AP50_theo_lop"]}'
          f'  ({kq["phut"]} phút, {kq["epoch_chay"]} epoch)')
    return kq

for v in range(1, 5):
    print(f'=== yolov8s-obb  vòng {v}  hạt 0 ===')
    mot_luot_khung('yolov8s-obb', v, 0)


giải nén khung 15 s
=== yolov8s-obb  vòng 1  hạt 0 ===
  bỏ qua, đã có: yolov8s-obb_v1_s0
=== yolov8s-obb  vòng 2  hạt 0 ===
  bỏ qua, đã có: yolov8s-obb_v2_s0
=== yolov8s-obb  vòng 3  hạt 0 ===
  nối tiếp từ last.pt: yolov8s-obb_v3_s0
Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ds_obb_v3/data.yaml, degrees=15, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, 

In [5]:
# 6. Gom + nhắc việc phải làm ở máy
import pandas as pd
rows = []
for v in range(K):
    p = f'{KQ}/ortho_tc_yolo26s-obb_v{v}_s0.json'
    if os.path.exists(p):
        d = json.load(open(p))
        rows.append(dict(vong=v, o_train=d['o_train'], o_test=d['o_test'], epoch=d['epoch_chay'],
                         truoc=(d['truoc'] or {}).get('mAP50'), sau=(d['sau'] or {}).get('mAP50')))
if rows:
    df = pd.DataFrame(rows); df['chenh'] = df['sau'] - df['truoc']
    print(df.to_string(index=False)); df.to_csv(f'{KQ}/tong_hop_ortho_tc.csv', index=False)

print('''
TẢI VỀ MÁY:
  colab_C/ket_qua/runs_ortho/yolo26s-obb_v*_s0/weights/best.pt  -> ket_qua/colab_C_ket_qua/runs_ortho/
  colab_C/ket_qua/ortho_tc_*.json                               -> ket_qua/colab_C_ket_qua/
  colab_B/ket_qua/yolov8s-obb_v*_s0.json + runs/                -> ket_qua/colab_B_ket_qua/

RỒI CHẠY Ở MÁY (mAP trên ô test chỉ là tín hiệu; con số của bài là cấp tấm):
  py -3.13 ma/quet_ortho_oof.py --runs ket_qua/colab_C_ket_qua/runs_ortho \\
         --out ket_qua/ortho_xoay270_tinhchinh_ds.json
  py -3.13 ma/gan_tam_pin_4lop.py --nguon ket_qua/ortho_xoay270_tinhchinh_ds.json
So với nhánh ortho chưa tinh chỉnh (R 0,834 cấp tấm, out-of-fold) và nhánh khung (R 0,924).
''')


 vong  o_train  o_test  epoch  truoc    sau  chenh
    0      139      44     54 0.4551 0.5541 0.0990
    1      133       0     36    NaN    NaN    NaN
    2      115      24     46 0.4962 0.5819 0.0857
    3      116      15     55 0.4289 0.7100 0.2811
    4      128      52     55 0.4306 0.7500 0.3194

TẢI VỀ MÁY:
  colab_C/ket_qua/runs_ortho/yolo26s-obb_v*_s0/weights/best.pt  -> ket_qua/colab_C_ket_qua/runs_ortho/
  colab_C/ket_qua/ortho_tc_*.json                               -> ket_qua/colab_C_ket_qua/
  colab_B/ket_qua/yolov8s-obb_v*_s0.json + runs/                -> ket_qua/colab_B_ket_qua/

RỒI CHẠY Ở MÁY (mAP trên ô test chỉ là tín hiệu; con số của bài là cấp tấm):
  py -3.13 ma/quet_ortho_oof.py --runs ket_qua/colab_C_ket_qua/runs_ortho \
         --out ket_qua/ortho_xoay270_tinhchinh_ds.json
  py -3.13 ma/gan_tam_pin_4lop.py --nguon ket_qua/ortho_xoay270_tinhchinh_ds.json
So với nhánh ortho chưa tinh chỉnh (R 0,834 cấp tấm, out-of-fold) và nhánh khung (R 0,924).

